In [3]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
import spacy
import json
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from spacy.training import Example
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def load_dataset():
    """Загружает датасет с Kaggle"""
    DATASET_DIR = Path.cwd() / "datasets" / "dataturks"
    DATASET_PATH = DATASET_DIR / "Entity Recognition in Resumes.json"
    
    if DATASET_DIR.exists():
        print("Dataset already loaded")
    else:
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files('dataturks/resume-entities-for-ner', 
                                   path=DATASET_DIR, quiet=False, unzip=True)
        print("Dataset downloaded")
    
    return DATASET_PATH

In [3]:
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    """Преобразует данные Dataturks в формат Spacy"""
    try:
        training_data = []
        
        with open(dataturks_JSON_FilePath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            data = json.loads(line)
            text = data['content']
            entities = []
            
            for annotation in data['annotation']:
                point = annotation['points'][0]
                labels = annotation['label']
                
                if not isinstance(labels, list):
                    labels = [labels]

                for label in labels:
                    entities.append((point['start'], point['end'] + 1, label))
            
            training_data.append((text, {"entities": entities}))
        
        print(f"Loaded {len(training_data)} samples")
        return training_data
    except Exception as e:
        print(f"Unable to process {dataturks_JSON_FilePath}")
        print(f"Error: {str(e)}")
        return None

def remove_overlapping_entities(data):
    """Удаляет перекрывающиеся сущности из данных"""
    cleaned_data = []
    conflict_count = 0
    
    for text, annotations in data:
        entities = annotations.get("entities", [])
        
        if not entities:
            cleaned_data.append((text, annotations))
            continue
        
        # Сортируем сущности по начальной позиции
        entities.sort(key=lambda x: (x[0], x[1]))
        
        # Убираем перекрывающиеся сущности
        non_overlapping = []
        prev_end = -1
        
        for start, end, label in entities:
            # Пропускаем некорректные сущности
            if start >= end or start < 0 or end > len(text):
                continue
            
            # Пропускаем перекрывающиеся сущности
            if start < prev_end:
                conflict_count += 1
                continue
            
            non_overlapping.append((start, end, label))
            prev_end = end
        
        cleaned_data.append((text, {"entities": non_overlapping}))
    
    if conflict_count > 0:
        print(f"Removed {conflict_count} overlapping entities")
    
    return cleaned_data

def clean_entity_spans(data):
    """Очищает пробелы в начале и конце сущностей"""
    cleaned_data = []
    
    for text, annotations in data:
        entities = annotations.get("entities", [])
        valid_entities = []
        
        for start, end, label in entities:
            # Убираем пробелы в начале
            while start < len(text) and text[start].isspace():
                start += 1
            
            # Убираем пробелы в конце
            while end > 0 and text[end-1].isspace():
                end -= 1
            
            # Проверяем, что span не стал пустым
            if start < end and start >= 0 and end <= len(text):
                valid_entities.append((start, end, label))
        
        cleaned_data.append((text, {"entities": valid_entities}))
    
    return cleaned_data

In [4]:
dataset_path = load_dataset()
raw_data = convert_dataturks_to_spacy(dataset_path)

Dataset already loaded
Loaded 220 samples


In [5]:
data = remove_overlapping_entities(raw_data)
data = clean_entity_spans(data)

Removed 112 overlapping entities


In [6]:
len(data)

220

In [7]:
train_data, test_data = train_test_split(data, test_size=0.1, random_state=42, shuffle=True)

In [2]:
def prepare_model(train_data):
    """Подготавливает модель Spacy для обучения"""
    nlp = spacy.blank('en')
    
    if 'ner' not in nlp.pipe_names:
        ner = nlp.add_pipe('ner', last=True)
    
    all_labels = set()
    for _, annotations in train_data:
        for ent in annotations.get('entities'):
            ner.add_label(ent[2])
            all_labels.add(ent[2])
    
    print(f"Labels to train: {sorted(all_labels)}")
    return nlp, list(all_labels)

def save_model(nlp, model_name="spacy_resume_ner_model"):
    """Сохраняет обученную модель"""
    output_dir = Path.cwd() / model_name
    nlp.to_disk(output_dir)
    print(f"\nModel saved to: {output_dir}")
    return output_dir

def load_saved_model(model_path):
    """Загружает сохраненную модель"""
    if Path(model_path).exists():
        nlp = spacy.load(model_path)
        print(f"Model loaded from: {model_path}")
        return nlp
    else:
        print(f"Model not found at: {model_path}")
        return None

In [9]:
def train_model(nlp, train_data, n_iter=10):
    """Обучает модель NER"""
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    
    with nlp.disable_pipes(*other_pipes):
        optimizer = nlp.initialize()
        losses_history = []
        
        for itn in range(n_iter):
            print(f"\nStarting iteration {itn + 1}/{n_iter}")
            random.shuffle(train_data)
            losses = {}
            
            for text, annotations in train_data:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                
                nlp.update(
                    [example],
                    drop=0.2,
                    sgd=optimizer,
                    losses=losses
                )
            
            print(f"Losses: {losses}")
            losses_history.append(losses.get('ner', 0))
    
    return nlp, losses_history

In [10]:
nlp, labels = prepare_model(train_data)

Labels to train: ['College Name', 'Companies worked at', 'Degree', 'Designation', 'Email Address', 'Graduation Year', 'Location', 'Name', 'Skills', 'UNKNOWN', 'Years of Experience']


In [11]:
nlp, losses_history = train_model(nlp, train_data, n_iter=10)



Starting iteration 1/10


c:\project\resume_ner\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Kavitha K
Senior System Engineer - Infosys Limited..." with entities "[(0, 9, 'Name'), (10, 32, 'Designation'), (35, 50,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\project\resume_ner\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Puneet Singh
Associate Software Engineer

Bengalur..." with entities "[(0, 12, 'Name'), (13, 40, 'Designation'), (42, 51...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
c:\project\resume_ner\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some enti

Losses: {'ner': np.float32(10507.0205)}

Starting iteration 2/10
Losses: {'ner': np.float32(5425.459)}

Starting iteration 3/10
Losses: {'ner': np.float32(4215.1567)}

Starting iteration 4/10
Losses: {'ner': np.float32(3398.0527)}

Starting iteration 5/10
Losses: {'ner': np.float32(3124.293)}

Starting iteration 6/10
Losses: {'ner': np.float32(2885.2993)}

Starting iteration 7/10
Losses: {'ner': np.float32(2776.7056)}

Starting iteration 8/10
Losses: {'ner': np.float32(2504.6948)}

Starting iteration 9/10
Losses: {'ner': np.float32(2493.2524)}

Starting iteration 10/10
Losses: {'ner': np.float32(2451.2766)}


In [12]:
save_model(nlp)


Model saved to: c:\project\resume_ner\notebooks\spacy_resume_ner_model


WindowsPath('c:/project/resume_ner/notebooks/spacy_resume_ner_model')

In [13]:
def evaluate_model(nlp, test_data):
    """Оценивает модель на тестовых данных"""
    print("\n" + "="*50)
    print("MODEL EVALUATION")
    print("="*50)
    
    # Token-level evaluation
    all_true = []
    all_pred = []
    
    # Entity-level evaluation
    entity_stats = {}
    
    for text, annot in test_data:
        doc = nlp(text)
        true_entities = annot.get("entities", [])
        
        # Token-level labels
        y_true = ['O'] * len(doc)
        y_pred = ['O'] * len(doc)
        
        # True labels
        for start, end, label in true_entities:
            entity_tokens = []
            for token in doc:
                if token.idx >= start and token.idx + len(token.text) <= end:
                    entity_tokens.append(token)
            
            if entity_tokens:
                for i, token in enumerate(entity_tokens):
                    if i == 0:
                        y_true[token.i] = f'B-{label}'
                    else:
                        y_true[token.i] = f'I-{label}'
        
        # Predicted labels
        for i, token in enumerate(doc):
            if token.ent_iob_ == 'O':
                y_pred[i] = 'O'
            else:
                y_pred[i] = f'{token.ent_iob_}-{token.ent_type_}'
        
        all_true.extend(y_true)
        all_pred.extend(y_pred)
        
        # Entity-level statistics
        true_entity_set = set((start, end, label) for start, end, label in true_entities)
        pred_entity_set = set((ent.start_char, ent.end_char, ent.label_) for ent in doc.ents)
        
        for start, end, label in true_entities:
            if label not in entity_stats:
                entity_stats[label] = {'tp': 0, 'fp': 0, 'fn': 0}
        
        for ent in doc.ents:
            if ent.label_ not in entity_stats:
                entity_stats[ent.label_] = {'tp': 0, 'fp': 0, 'fn': 0}
        
        # Calculate TP, FP, FN
        for true_entity in true_entity_set:
            if true_entity in pred_entity_set:
                entity_stats[true_entity[2]]['tp'] += 1
            else:
                entity_stats[true_entity[2]]['fn'] += 1
        
        for pred_entity in pred_entity_set:
            if pred_entity not in true_entity_set:
                entity_stats[pred_entity[2]]['fp'] += 1
    
    # Calculate metrics
    print("\nToken-level Metrics:")
    print(classification_report(all_true, all_pred, zero_division=0))
    
    token_accuracy = accuracy_score(all_true, all_pred)
    print(f"Token Accuracy: {token_accuracy:.4f}")
    
    print("\n" + "="*50)
    print("Entity-level Metrics:")
    print("="*50)
    
    results_df = []
    for entity_type, stats in entity_stats.items():
        tp = stats['tp']
        fp = stats['fp']
        fn = stats['fn']
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results_df.append({
            'Entity': entity_type,
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'Precision': f"{precision:.4f}",
            'Recall': f"{recall:.4f}",
            'F1-Score': f"{f1:.4f}"
        })
    
    if results_df:
        df = pd.DataFrame(results_df)
        print(df.to_string(index=False))
    else:
        print("No entities found")
    
    return {
        'token_accuracy': token_accuracy,
        'classification_report': classification_report(all_true, all_pred, output_dict=True, zero_division=0),
        'entity_stats': entity_stats
    }

In [14]:
evaluation_results = evaluate_model(nlp, test_data)


MODEL EVALUATION

Token-level Metrics:
                       precision    recall  f1-score   support

       B-College Name       0.50      0.45      0.47        31
B-Companies worked at       0.40      0.34      0.37        50
             B-Degree       0.66      0.63      0.64        30
        B-Designation       0.63      0.35      0.45        49
      B-Email Address       0.81      0.94      0.87        18
    B-Graduation Year       0.62      0.37      0.47        27
           B-Location       0.62      0.35      0.45        43
               B-Name       1.00      1.00      1.00        22
             B-Skills       0.75      0.34      0.47        44
            B-UNKNOWN       0.00      0.00      0.00         1
B-Years of Experience       0.00      0.00      0.00         3
       I-College Name       0.56      0.31      0.40       105
I-Companies worked at       0.40      0.11      0.17        56
             I-Degree       0.78      0.68      0.73        88
        I-Desi

In [4]:
def test_model(nlp, test_texts=None):
    """Тестирует модель на произвольных текстах"""
    if test_texts is None:
        test_texts = [
            "John Smith worked at Google as a Senior Software Engineer from 2018 to 2022." +
            "Microsoft hired Sarah Johnson who graduated from Stanford University in 2015." +
            "Python, Java, and TensorFlow are required skills for this machine learning position." +
            "Experience with AWS, Docker, and Kubernetes is preferred for DevOps roles." +
            "Bachelor's degree in Computer Science from MIT with 5 years experience at Amazon." +
            "Artem Makarenkov is coding on Python"
        ]
    
    print("\n" + "="*50)
    print("MODEL TESTING")
    print("="*50)
    
    results = []
    for text in test_texts:
        doc = nlp(text)
        entities = [(ent.text, ent.label_, ent.start_char, ent.end_char) 
                   for ent in doc.ents]
        
        result = {
            'text': text,
            'entities': entities,
            'formatted': f"Text: {text}\nEntities:"
        }
        
        if entities:
            for ent_text, ent_label, start, end in entities:
                result['formatted'] += f"\n  '{ent_text}' -> {ent_label}"
        else:
            result['formatted'] += "\n  No entities found"
        
        results.append(result)
        print(result['formatted'] + "\n")
    
    return results

In [ ]:
nlp = load_saved_model(Path.cwd().parent / "resume_ner_model")
test_model(nlp)

Model loaded from: c:\project\resume_ner\resume_ner_model_old

MODEL TESTING
Text: John Smith worked at Google as a Senior Software Engineer from 2018 to 2022.Microsoft hired Sarah Johnson who graduated from Stanford University in 2015.Python, Java, and TensorFlow are required skills for this machine learning position.Experience with AWS, Docker, and Kubernetes is preferred for DevOps roles.Bachelor's degree in Computer Science from MIT with 5 years experience at Amazon.Artem Makarenkov is coding on Python
Entities:
  'John Smith' -> Name



[{'text': "John Smith worked at Google as a Senior Software Engineer from 2018 to 2022.Microsoft hired Sarah Johnson who graduated from Stanford University in 2015.Python, Java, and TensorFlow are required skills for this machine learning position.Experience with AWS, Docker, and Kubernetes is preferred for DevOps roles.Bachelor's degree in Computer Science from MIT with 5 years experience at Amazon.Artem Makarenkov is coding on Python",
  'entities': [('John Smith', 'Name', 0, 10)],
  'formatted': "Text: John Smith worked at Google as a Senior Software Engineer from 2018 to 2022.Microsoft hired Sarah Johnson who graduated from Stanford University in 2015.Python, Java, and TensorFlow are required skills for this machine learning position.Experience with AWS, Docker, and Kubernetes is preferred for DevOps roles.Bachelor's degree in Computer Science from MIT with 5 years experience at Amazon.Artem Makarenkov is coding on Python\nEntities:\n  'John Smith' -> Name"}]